<a href="https://colab.research.google.com/github/Andreas-Lukito/Stock_Sentiment_Analysis/blob/dev%2Fandreas/notebooks/04_FinBERT_categorical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinBERT for Predicting News Sentiment

## Install Libraries

In [31]:
! pip install contractions emoji gensim optuna torch matplotlib
! pip install wandb

## Iport Libraries

In [32]:
# Common Python Libraries
import numpy as np
import pandas as pd
import os
import sys
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import random
from datasets import Dataset

# Deep Learning Libraries
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer, EarlyStoppingCallback
from torch.optim import Adam, AdamW
from accelerate import Accelerator

# Data Preprocessing
from sklearn.model_selection import train_test_split

## Download nltk dependencies
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

# Model metrics
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, RocCurveDisplay, f1_score, precision_score, recall_score, accuracy_score
from itertools import cycle
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Model Tracking
import wandb

# Google Colab Setup
from google.colab import drive
drive.mount('/content/drive')
project_path = "/content/drive/MyDrive/stock_news_sentiment_analysis"

# project_path = "../"

# Project Seed for Reproducability
SEED = random.randint(0, 2**32 - 1)  # Random integer between 0 and 2^32-1
print(f"seed: {SEED}")

model_name = "ProsusAI/finbert"

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
seed: 3175870115


## Choose Device

In [33]:
# Detect available device
if torch.cuda.is_available():
    # check if ROCm backend is active
    if torch.version.hip is not None:
        backend = "ROCm"
    else:
        backend = "CUDA"

    device = torch.device("cuda")
    print(f"PyTorch is using GPU: {torch.cuda.get_device_name(0)}")
    print(f"Backend: {backend}")
else:
    device = torch.device("cpu")
    print("PyTorch is not using GPU — running on CPU")

PyTorch is using GPU: NVIDIA A100-SXM4-80GB
Backend: CUDA


## Import Data

In [ ]:
before_date = "2025-11"

# Data path
categorized_data_path = os.path.join(project_path,f"news_cache/catgorized_data/categorized_news_data2.csv")

# Import Data
news_data = pd.read_csv(filepath_or_buffer=categorized_data_path, sep=',')

In [ ]:
news_data.head()

In [ ]:
news_data.isna().sum()

In [ ]:
news_data = news_data.dropna(subset=["sentiment"])

In [ ]:
news_data.isna().sum()

In [ ]:
value_maps = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

news_data["categorical_sentiment_3_class_value_maps"] = news_data["categorical_sentiment_3_class"].map(value_maps)

## Split the data to Train, Test, and Validation

In [ ]:
test_size = 0.20
val_size = 0.50

# Splitting the data into train and temp (which will be further split into validation and test)
train_df, test_df = train_test_split(news_data, test_size=test_size, random_state=SEED, stratify=news_data["categorical_sentiment_3_class_value_maps"])

# Splitting train into validation and test sets
val_df, test_df = train_test_split(test_df, test_size=val_size, random_state=SEED, stratify=test_df["categorical_sentiment_3_class_value_maps"])

In [ ]:
train_df.shape, test_df.shape, val_df.shape

In [ ]:
train_df = train_df[["clean_text", "categorical_sentiment_3_class_value_maps"]]
train_df = train_df.rename(columns = {"clean_text": "text", "categorical_sentiment_3_class_value_maps":"labels"})

val_df = val_df[["clean_text", "categorical_sentiment_3_class_value_maps"]]
val_df = val_df.rename(columns = {"clean_text": "text", "categorical_sentiment_3_class_value_maps":"labels"})

test_df = test_df[["clean_text","categorical_sentiment_3_class_value_maps"]]
test_df = test_df.rename(columns = {"clean_text": "text", "categorical_sentiment_3_class_value_maps":"labels"})

## Data Preprocessing

### Convert Dataframe to Dataset for the Model Trainer

In [ ]:
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)

### Tokenizer for the text

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples, tokenizer=tokenizer):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding=True,
        max_length=128   # keep this
    )

In [ ]:
train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

In [ ]:
# remove unwanted columns
train_ds = train_ds.remove_columns(["text", "__index_level_0__"])
val_ds = val_ds.remove_columns(["text", "__index_level_0__"])
test_ds = test_ds.remove_columns(["text", "__index_level_0__"])

## Train Model

### Model Arguments

In [ ]:
model_checkpoint_directory = os.path.join(project_path, "model_checkpoints/finbert_categorical")
os.makedirs(model_checkpoint_directory, exist_ok=True)
learning_rate=2e-5

training_args = TrainingArguments(
    output_dir=model_checkpoint_directory,
    report_to="wandb",
    run_name=f"finbert_categorical_{learning_rate}",

    eval_strategy="epoch",
    save_strategy="epoch",

    num_train_epochs=1000,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,

    learning_rate=learning_rate,

    load_best_model_at_end=True,
    metric_for_best_model="f1_score",
    greater_is_better=True,

    save_total_limit=2,

    seed=SEED
)

### Model Metrics for Training

In [ ]:
def model_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # convert logits → probabilities
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()

    f1 = f1_score(labels, predictions, average='weighted', zero_division=0)
    precision = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall = recall_score(labels, predictions, average='weighted', zero_division=0)
    accuracy = accuracy_score(labels, predictions)

    try:
        roc_auc = roc_auc_score(labels, probs, multi_class='ovr')
    except:
        roc_auc = None

    return {
        "f1_score": f1,
        "precision": precision,
        "recall": recall,
        "accuracy": accuracy,
        "roc_auc": roc_auc
    }

### wandb setup

In [ ]:
wandb.login()

# set the wandb project where this run will be logged
os.environ["WANDB_PROJECT"]="Thesis-Stock-Sentiment-Analysis"

# save your trained model checkpoint to wandb
os.environ["WANDB_LOG_MODEL"]="true"

# turn off watch to log faster
os.environ["WANDB_WATCH"]="false"

### Model Trainer

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3, # Since there are three classes ["Negative", "Neutral", "Positive"]
    ignore_mismatched_sizes=True
)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    compute_metrics = model_metrics
    ,callbacks=[EarlyStoppingCallback(early_stopping_patience=10)]
)

In [ ]:
train_ds.column_names

In [ ]:
trainer.train()

model_save_path = os.path.join(model_checkpoint_directory, "final_model")
os.makedirs(model_save_path, exist_ok=True)
trainer.save_model(model_save_path)

In [ ]:
# wandb.finish() # Moved to run after evaluation

## Model Evaluation

### Model Evaluation Function

In [ ]:
def evaluate_model_with_trainer(trainer, dataset):
    predictions_output = trainer.predict(dataset)
    logits = predictions_output.predictions
    labels = predictions_output.label_ids
    metrics = predictions_output.metrics

    preds = np.argmax(logits, axis=-1)

    # Generate classification report
    report = classification_report(labels, preds, zero_division=0)

    # Generate confusion matrix
    cm = confusion_matrix(labels, preds)

    # For ROC AUC, we need probabilities. The model outputs logits, so convert to probabilities.
    # Make sure to handle multi_class correctly for 'ovr'
    try:
        probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
        roc_auc = roc_auc_score(labels, probs, multi_class='ovr')
    except ValueError: # Handle cases where roc_auc_score might fail (e.g., single class in evaluation)
        roc_auc = None

    return report, cm, roc_auc, labels, preds

### Get the Model

In [ ]:
trained_model = trainer.model

# Re-instantiate Trainer for evaluation to ensure Accelerator is in a valid state
evaluation_trainer = Trainer(
    model=trained_model, # Use the best model obtained after training
    args=training_args,  # Use the same training arguments
    eval_dataset=test_ds, # Only need the test dataset for testing
    compute_metrics=model_metrics # Keep the same metrics computation
)

In [ ]:
classification_report_str, confusion_matrix_val, roc_auc_val, labels, preds = evaluate_model_with_trainer(evaluation_trainer, test_ds)

In [ ]:
print("========== Classification Report ==========")
print(classification_report_str)

print("========== Confusion Matrix ==========")
print(confusion_matrix_val)

print()
print("========== ROC_AUC ==========")
print(roc_auc_val)

In [ ]:
def plot_roc_curve(y_test, y_pred):

  n_classes = len(np.unique(y_test))
  y_test = label_binarize(y_test, classes=np.arange(n_classes))
  y_pred = label_binarize(y_pred, classes=np.arange(n_classes))

  # Compute ROC curve and ROC area for each class
  fpr = dict()
  tpr = dict()
  roc_auc = dict()
  for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test[:, i], y_pred[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

  # Compute micro-average ROC curve and ROC area
  fpr["micro"], tpr["micro"], _ = roc_curve(y_test.ravel(), y_pred.ravel())
  roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

  # First aggregate all false positive rates
  all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))

  # Then interpolate all ROC curves at this points
  mean_tpr = np.zeros_like(all_fpr)
  for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])

  # Finally average it and compute AUC
  mean_tpr /= n_classes

  fpr["macro"] = all_fpr
  tpr["macro"] = mean_tpr
  roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

  # Plot all ROC curves
  #plt.figure(figsize=(10,5))
  plt.figure(dpi=600)
  lw = 2
  plt.plot(fpr["micro"], tpr["micro"],
    label="micro-average ROC curve (area = {0:0.2f})".format(roc_auc["micro"]),
    color="deeppink", linestyle=":", linewidth=4,)

  plt.plot(fpr["macro"], tpr["macro"],
    label="macro-average ROC curve (area = {0:0.2f})".format(roc_auc["macro"]),
    color="navy", linestyle=":", linewidth=4,)

  colors = cycle(["aqua", "darkorange", "darkgreen", "yellow", "blue"])
  for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=lw,
        label="ROC curve of class {0} (area = {1:0.2f})".format(i, roc_auc[i]),)

  plt.plot([0, 1], [0, 1], "k--", lw=lw)
  plt.xlim([0.0, 1.0])
  plt.ylim([0.0, 1.05])
  plt.xlabel("False Positive Rate")
  plt.ylabel("True Positive Rate")
  plt.title("Receiver Operating Characteristic (ROC) curve")
  plt.legend()

In [ ]:
plot_roc_curve(labels, preds)